In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q16/annotated/checkpoints/post_cell_2.pickle

trying: ['tpch_parent']
me:  0
trying: ['orig_output']
me:  4
trying: ['STORAGE_OPTS']
me:  0
trying: ['load_part']
me:  1
trying: ['part']
me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['supplier']
me:  3
trying: ['load_supplier']
me:  3
trying: ['load_partsupp']
me:  2
trying: ['pd']
me:  0
trying: ['partsupp']
me:  2
Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable load_part
Declaring variable part
Declaring variable load_partsupp
Declaring variable partsupp
Declaring variable supplier
Declaring variable load_supplier
Declaring variable orig_output


me:  2
Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable load_part
Declaring variable part
Declaring variable load_partsupp
Declaring variable partsupp
Declaring variable supplier
Declaring variable load_supplier
Declaring variable orig_output


In [4]:
%%RecordEvent
%%time
### cell 3 ###

### cell 3 (optimized for cudf)
# restrict part to only the columns and rows we need
sizes = [49, 14, 23, 45, 19, 3, 36, 9]
part_filtered = (
    part
    [(part.P_BRAND != "Brand#45")
     & (~part.P_TYPE.str.contains(r"^MEDIUM POLISHED"))
     & part.P_SIZE.isin(sizes)]
    [["P_PARTKEY", "P_BRAND", "P_TYPE", "P_SIZE"]]
)

# restrict partsupp to the columns we need
partsupp_filtered = partsupp[["PS_PARTKEY", "PS_SUPPKEY"]]

# inner‐join part and partsupp
total = (
    part_filtered
    .merge(partsupp_filtered,
           left_on="P_PARTKEY", right_on="PS_PARTKEY",
           how="inner")[["P_BRAND", "P_TYPE", "P_SIZE", "PS_SUPPKEY"]]
)

# build a deduplicated list of suppliers to exclude
supplier_keys = (
    supplier
    [supplier.S_COMMENT.str.contains(r"Customer(\S|\s)*Complaints")]
    ["S_SUPPKEY"].drop_duplicates()
)

# filter out any PS_SUPPKEY that appears in that supplier list
total = total[~total.PS_SUPPKEY.isin(supplier_keys)]

# group, count unique suppliers, rename, and sort
total = (
    total
    .groupby(["P_BRAND", "P_TYPE", "P_SIZE"],
              as_index=False, sort=False)
    .agg({"PS_SUPPKEY": "nunique"})
    .rename(columns={"PS_SUPPKEY": "SUPPLIER_CNT"})
    .sort_values(
        by=["SUPPLIER_CNT", "P_BRAND", "P_TYPE", "P_SIZE"],
        ascending=[False, True, True, True]
    )
)


CPU times: user 83.4 ms, sys: 15 ms, total: 98.4 ms
Wall time: 102 ms


In [7]:
total

,P_BRAND,P_TYPE,P_SIZE,SUPPLIER_CNT
10509,Brand#41,MEDIUM BRUSHED TIN,3,28
4712,Brand#54,STANDARD BRUSHED COPPER,14,27
4783,Brand#11,STANDARD BRUSHED TIN,23,24
4369,Brand#11,STANDARD BURNISHED BRASS,36,24
1353,Brand#15,MEDIUM ANODIZED NICKEL,3,24
...,...,...,...,...
16506,Brand#52,MEDIUM BRUSHED BRASS,45,3
7362,Brand#53,MEDIUM BRUSHED TIN,45,3
419,Brand#54,ECONOMY POLISHED BRASS,9,3
17098,Brand#55,PROMO PLATED BRASS,19,3


In [5]:
total.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 18314 entries, 10509 to 12445
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   P_BRAND       18314 non-null  object
 1   P_TYPE        18314 non-null  object
 2   P_SIZE        18314 non-null  int64
 3   SUPPLIER_CNT  18314 non-null  int32
dtypes: int32(1), int64(1), object(2)
memory usage: 1012.1+ KB


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q16/rewritten/o4_mini_high/checkpoints/post_cell_3_try_2.pickle

migration speed (bps): 805971651.2904382
---------------------------
variables to migrate:
DATA_ROOT 80
STORAGE_OPTS 64
load_part 144
part 48
load_supplier 144
partsupp 48
sizes 120
supplier 48
load_partsupp 144
pd 72
orig_output 16
partsupp_filtered 48
tpch_parent 54
total 48
supplier_keys 48
part_filtered 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q16/rewritten/o4_mini_high/checkpoints/post_cell_3_try_2.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['partsupp']
Future variables ['part']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['part', 'partsupp']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['partsupp_filtered', 'total', 'sizes', 'supplier_keys', 'part_filtered']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q16/opt_cell_exec_info_3_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q16/annotated/checkpoints/post_cell_3.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['part_filtered']
me:  5
trying: ['DATA_ROOT']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['load_part']
me:  1
trying: ['part']
me:  1
trying: ['load_supplier']
me:  3
trying: ['supplier_filtered']
me:  5
trying: ['partsupp']
me:  2
trying: ['supplier']
me:  3
trying: ['load_partsupp']
me:  2
trying: ['pd']
me:  0
trying: ['orig_output']
me:  4
trying: ['total']
me:  5
trying: ['tpch_parent']
me:  0
trying: ['partsupp_filtered']
me:  5


Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable pd
Declaring variable tpch_parent
Declaring variable load_part
Declaring variable part
Declaring variable partsupp
Declaring variable load_partsupp
Declaring variable load_supplier
Declaring variable supplier
Declaring variable orig_output
Declaring variable part_filtered
Declaring variable supplier_filtered
Declaring variable total
Declaring variable partsupp_filtered
